In [1]:
import os
import numpy as np
import networkx as nx
import pickle
import pandas as pd
from scipy.spatial import ConvexHull
from scipy import linalg
import porespy as ps
from skimage import draw

In [3]:
core_path = '../graph_single_mask/'

folders = ['ac_control', 'lat_control', 'vm_control', 'pc_control', 'dm_control']

interval_labels = ['[3π/8,5π/8]', '[π/4,3π/8]∪[5π/8,3π/4]', '[π/8,π/4]∪[3π/4,7π/8]', '[0,π/8]∪[7π/8,π]']


n = 5
label_map = {
    'ac_control': 0, # AC
    'dm_control': 1, # DM
    'lat_control': 2, # LAT
    'vm_control': 3, # VM
    'pc_control': 4 # PC
}
# Dictionaries to store data for each feature
id_data = {folder: [] for folder in folders}
number_of_edges_data = {folder: [] for folder in folders}
mean_branch_length_data = {folder: [] for folder in folders}
branch_density_data = {folder: [] for folder in folders}
mass_data = {folder: {i+1: [] for i in range(4)} for folder in folders}
directional_std_data = {folder: [] for folder in folders}
directional_entropy_data = {folder: [] for folder in folders}
orientation_order_data = {folder: [] for folder in folders}
circuity_data = {folder: [] for folder in folders}
mean_betweenness_centrality_data = {folder: [] for folder in folders}
graph_diameter_data = {folder: [] for folder in folders}
apl_data = {folder: [] for folder in folders}
assortativity_data = {folder: [] for folder in folders}
algebraic_connectivity_data = {folder: [] for folder in folders}
spectral_entropy_data = {folder: [] for folder in folders}
fractal_dimension_data = {folder: [] for folder in folders}
bending_energy_data = {folder: [] for folder in folders}

def average_bending_energy_curve(c, closed=False, eps=1e-12):
    """
    Compute the line integral of the squared curvature of a polyline:
        ∫ κ(s)^2 ds / len(c)
    using a discrete arclength-based scheme on edges/tangents.
    Parameters
    ----------
    c : (N, d) array_like
        Vertices of the curve (d=2 or d=3).
    closed : bool, default True
        If True, treat the curve as closed (cyclic indices).
        If False, treat as open and use only interior vertices.
    eps : float, default 1e-12
        Small clamp to avoid division by zero for tiny edges.
    Returns
    -------
    float
        Discrete approximation of ∫ κ^2 ds.
    Notes
    -----
    Let e_i = c_{i+1} - c_i, ℓ_i = ||e_i||, and t_{i+1/2} = e_i / ℓ_i.
    For a closed curve, the per-vertex contribution at vertex i is:
        ||t_{i+1/2} - t_{i-1/2}||^2 / ((ℓ_i + ℓ_{i-1})/2).
    For an open curve, the sum is taken over interior vertices only.
    """
    c = np.asarray(c, dtype=np.float64)
    if c.ndim != 2 or c.shape[1] < 2:
        raise ValueError("c must be an (N, d) array with d >= 2.")
    N, d = c.shape
    if N < 3:
        raise ValueError("Need at least 3 points to estimate curvature.")
    if closed:
        # Edges, lengths, and unit tangents on edges
        e = np.roll(c, -1, axis=0) - c # (N, d), edges i -> i+1
        ℓ = np.linalg.norm(e, axis=1)
        length = np.sum(ℓ)
        ℓ = np.maximum(ℓ, eps)
        t = e / ℓ[:, None] # unit tangents on edges
        # Δt at vertices (i): t_{i+1/2} - t_{i-1/2}
        Δt = t - np.roll(t, 1, axis=0) # (N, d)
        # Dual (Voronoi) cell length around vertex i
       
        avg_ℓ = 0.5 * (ℓ + np.roll(ℓ, 1)) # (N,)
        avg_ℓ = np.maximum(avg_ℓ, eps)
        # Sum of ||Δt||^2 / avg_ℓ
        energy = np.sum(np.einsum('ij,ij->i', Δt, Δt) / avg_ℓ)/length
        return float(energy)
    else:
        # Open curve: edges on [0..N-2]
        e = c[1:] - c[:-1] # (N-1, d)
        ℓ = np.linalg.norm(e, axis=1)
        length = np.sum(ℓ)
        ℓ = np.maximum(ℓ, eps)
        t = e / ℓ[:, None] # (N-1, d)
        # Interior vertices correspond to differences of edge tangents
        Δt = t[1:] - t[:-1] # (N-2, d)
        avg_ℓ = 0.5 * (ℓ[1:] + ℓ[:-1]) # (N-2,)
        avg_ℓ = np.maximum(avg_ℓ, eps)
        energy = np.sum(np.einsum('ij,ij->i', Δt, Δt) / avg_ℓ)/length
        return float(energy)

# Function to compute graph_to_binary_image
def graph_to_binary_image(graph, image_size=700, radius_scale=2.0):
    all_x = []
    all_y = []
    for s, e in graph.edges():
        pts = graph[s][e]['pts']
        if pts.ndim != 2 or pts.shape[1] < 2:
            continue
        all_x.extend(pts[:,1])
        all_y.extend(pts[:,0])
    if not all_x:
        return np.zeros((image_size, image_size), dtype=bool)
    x_min, x_max = min(all_x), max(all_x)
    y_min, y_max = min(all_y), max(all_y)
    extent = max(x_max - x_min, y_max - y_min) * 1.1
    if extent == 0:
        return np.zeros((image_size, image_size), dtype=bool)
    pixel_size = extent / (image_size - 1)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    im = np.zeros((image_size, image_size), dtype=bool)
    def scale_coord(coord, center, extent):
        return int(((coord - center) / extent + 0.5) * (image_size - 1))
    for s, e in graph.edges():
        pts = graph[s][e]['pts']
        if len(pts) < 2:
            continue
        xs = pts[:,1]
        ys = pts[:,0]
        radius = max(1, int(radius_scale))
        for i in range(len(pts) - 1):
            x1 = scale_coord(xs[i], x_center, extent)
            y1 = scale_coord(ys[i], y_center, extent)
            x2 = scale_coord(xs[i+1], x_center, extent)
            y2 = scale_coord(ys[i+1], y_center, extent)
            rr, cc = draw.line(y1, x1, y2, x2)
            valid = (0 <= rr) & (rr < image_size) & (0 <= cc) & (cc < image_size)
            rr, cc = rr[valid], cc[valid]
            for j in range(len(rr)):
                r, c = rr[j], cc[j]
                rr_disk, cc_disk = draw.disk((r, c), radius, shape=im.shape)
                im[rr_disk, cc_disk] = True
    return im

for folder in folders:
    folder_path = os.path.join(core_path, folder)
    if not os.path.exists(folder_path):
        print(f"Folder {folder_path} does not exist.")
        continue
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".pkl"):
            id_data[folder].append(file_name)
            file_path = os.path.join(folder_path, file_name)
            with open(file_path, "rb") as file:
                graph = pickle.load(file)
            # Initialize variables
            local_vectors = []
            local_lengths = []
            straight_line_lengths = []
            graph_branch_lengths = []
            all_points = []
            # Compute number of branches
            num_branches = len(graph.edges())
            number_of_edges_data[folder].append(num_branches)
            
            # Process edges
            for (s, e) in graph.edges():
                pts = graph[s][e]['pts']
                num_px = pts.shape[0]
                # Calculate branch length
                branch_length = np.sum(np.sqrt(np.sum(np.diff(pts, axis=0) ** 2, axis=1)))
                graph_branch_lengths.append(branch_length)
                all_points.extend(pts)
                # Process points for other metrics
                if num_px > n + 1:
                    red_pts = [(pts[0, 1], pts[0, 0])]
                    for i in range((num_px - 1) // n):
                        idx = n * (i + 1)
                        if idx < num_px:
                            red_pts.append((pts[idx, 1], pts[idx, 0]))
                    if (pts[-1, 1], pts[-1, 0]) != red_pts[-1]:
                        red_pts.append((pts[-1, 1], pts[-1, 0]))
                    red_pts = np.array(red_pts)
                    segment_length = 0
                    for i in range(len(red_pts) - 1):
                        x0, y0 = red_pts[i]
                        x1, y1 = red_pts[i + 1]
                        dx, dy = x1 - x0, y1 - y0
                        length = np.sqrt(dx**2 + dy**2)
                        if length > 0:
                            segment_length += length
                            local_vectors.append((dx / length, dy / length))
                            local_lengths.append(length)
                    x0, y0 = red_pts[0]
                    x1, y1 = red_pts[-1]
                    straight_length = np.sqrt((x1 - x0)**2 + (y1 - y0)**2)
                    straight_line_lengths.append(straight_length)
                    
            # Compute mean branch length per graph
            mean_branch = np.mean(graph_branch_lengths) if graph_branch_lengths else 0
            mean_branch_length_data[folder].append(mean_branch)
            
            # Calculate edge density
            all_points = np.array(all_points)
            total_branch_length = sum(graph_branch_lengths)
            if len(np.unique(all_points, axis=0)) >= 3:
                hull = ConvexHull(all_points)
                area = hull.volume
            else:
                area = 0
                print(f"Warning: Not enough unique points to compute convex hull for {file_name}. Setting area to 0.")
            branch_density = total_branch_length / area if area > 0 else 0
            branch_density_data[folder].append(branch_density)
            if len(local_vectors) == 0:
                # Assign default values for vector-based metrics
                mass_data[folder][1].append(0)
                mass_data[folder][2].append(0)
                mass_data[folder][3].append(0)
                mass_data[folder][4].append(0)
                directional_std_data[folder].append(0)
                directional_entropy_data[folder].append(0)
                orientation_order_data[folder].append(0)
                circuity_data[folder].append(1)
            else:
                # Compute angles
                angles = np.arctan2(
                    [dy for dx, dy in local_vectors],
                    [dx for dx, dy in local_vectors]
                ) % np.pi
                angles_full = 2 * angles
                edge_weights = np.array(local_lengths)
                edge_weights /= np.sum(edge_weights)
                
                # Compute circular mean
                
                p = 1
                C_p = np.sum(edge_weights * np.cos(p * angles_full))
                S_p = np.sum(edge_weights * np.sin(p * angles_full))
                theta_mean_full = np.arctan2(S_p, C_p) % (2 * np.pi)
                theta_mean = theta_mean_full / 2
                R_p_mean = np.sqrt(C_p**2 + S_p**2)
                # Circular Standard Deviation
                circular_std = np.sqrt(-2 * np.log(R_p_mean)) if R_p_mean > 0 else 0
                directional_std_data[folder].append(circular_std)
                # Rotate angles to align mean to π/2
                rotated_angles = (angles - theta_mean + np.pi/2) % np.pi
                
                # Define interval masks for 4 Quartiles
                mask_4 = ((rotated_angles >= 0) & (rotated_angles <= np.pi / 8)) | \
                         ((rotated_angles >= 7 * np.pi / 8) & (rotated_angles <= np.pi))
                mask_3 = ((rotated_angles > np.pi / 8) & (rotated_angles <= np.pi / 4)) | \
                         ((rotated_angles >= 3 * np.pi / 4) & (rotated_angles < 7 * np.pi / 8))
                mask_2 = ((rotated_angles > np.pi / 4) & (rotated_angles <= 3 * np.pi / 8)) | \
                         ((rotated_angles >= 5 * np.pi / 8) & (rotated_angles < 3 * np.pi / 4))
                mask_1 = (rotated_angles > 3 * np.pi / 8) & (rotated_angles < 5 * np.pi / 8)

                
                # Compute mass in each interval
                mass_1 = np.sum(edge_weights * mask_1)
                mass_2 = np.sum(edge_weights * mask_2)
                mass_3 = np.sum(edge_weights * mask_3)
                mass_4 = np.sum(edge_weights * mask_4)
                
                # Entropy, Orientation-Order, and Circuity
                n_bins = 18
                bin_edges = np.linspace(0, np.pi, n_bins + 1)
                hist_weighted, _ = np.histogram(rotated_angles, bins=bin_edges, weights=edge_weights, density=False)
                total_weight = np.sum(hist_weighted)
                p_weighted = hist_weighted / total_weight if total_weight > 0 else np.zeros_like(hist_weighted)
                H_weighted = -np.sum([p * np.log(p) if p > 0 else 0 for p in p_weighted])
                H_max = np.log(18)
                phi = 1 - (H_weighted / H_max)**2 if H_max != 0 else 0
                L_net = sum(local_lengths)
                L_g = sum(straight_line_lengths)
                circuity = L_net / L_g if L_g > 0 else 1
                
                # Store metrics
                mass_data[folder][1].append(mass_1)
                mass_data[folder][2].append(mass_2)
                mass_data[folder][3].append(mass_3)
                mass_data[folder][4].append(mass_4)
                directional_entropy_data[folder].append(H_weighted)
                orientation_order_data[folder].append(phi)
                circuity_data[folder].append(circuity)
                
            # Betweenness Centrality (Mean)
            try:
                bc = nx.betweenness_centrality(graph, weight='weight', normalized=True)
                mean_bc = np.mean(list(bc.values()))
                mean_betweenness_centrality_data[folder].append(mean_bc)
            except Exception as e:
                print(f"Betweenness centrality error in {file_name}: {e}")
                mean_betweenness_centrality_data[folder].append(0)
                
            # Graph Diameter
            try:
                diameter = nx.diameter(graph, weight='weight')
                graph_diameter_data[folder].append(diameter)
            except Exception as e:
                print(f"Diameter error in {file_name}: {e}")
                graph_diameter_data[folder].append(0)
                
            # Average Path Length
            try:
                apl = nx.average_shortest_path_length(graph, weight='weight')
                apl_data[folder].append(apl)
            except Exception as e:
                print(f"Average shortest path error in {file_name}: {e}")
                apl_data[folder].append(0)
                
            # Assortativity
            try:
                if graph.number_of_edges() < 2:
                    assortativity = 0
                else:
                    assortativity = nx.degree_assortativity_coefficient(graph, weight='weight')
                    assortativity = 0 if np.isnan(assortativity) else assortativity
                assortativity_data[folder].append(assortativity)
            except Exception as e:
                print(f"Assortativity error in {file_name}: {e}")
                assortativity_data[folder].append(0)
            # Algebraic Connectivity
            try:
                alg_conn = nx.algebraic_connectivity(graph, weight='weight', method='tracemin_pcg')
                algebraic_connectivity_data[folder].append(alg_conn)
            except Exception as e:
                print(f"Algebraic connectivity error in {file_name}: {e}")
                algebraic_connectivity_data[folder].append(0)
            # Spectral Entropy
            try:
                L = nx.laplacian_matrix(graph, weight='weight').astype(float).toarray()
                eigenvalues = linalg.eigvalsh(L)
                eigenvalues = np.maximum(eigenvalues, 1e-12)
                probs = eigenvalues / np.sum(eigenvalues)
                spectral_entropy = -np.sum(probs * np.log(probs))
                spectral_entropy_data[folder].append(spectral_entropy)
            except Exception as e:
                print(f"Spectral entropy error in {file_name}: {e}")
                spectral_entropy_data[folder].append(0)
            # Fractal Dimension
            try:
                im = graph_to_binary_image(graph, image_size=700, radius_scale=2.0)
                box_sizes = np.logspace(np.log10(5), np.log10(50), num=5, dtype=int)
                boxcount = ps.metrics.boxcount(im, bins=box_sizes)
                stable_region = (boxcount.size >= 5) & (boxcount.size <= 50)
                fd = np.mean(boxcount.slope[stable_region]) if np.any(stable_region) else 0.0
                fractal_dimension_data[folder].append(fd)
            except Exception as e:
                print(f"Fractal dimension error in {file_name}: {e}")
                fractal_dimension_data[folder].append(0)

            # Add average bending energy
            # Extract branch polylines
            branch_polylines = []
            visited_edges = set()
            for node in list(graph.nodes()):
                if graph.degree[node] != 2: # Key nodes: tips (deg=1) or branch points (deg>2)
                    for neighbor in list(graph.neighbors(node)):
                        current = node
                        next_node = neighbor
                        edge = tuple(sorted((current, next_node)))
                        if edge in visited_edges:
                            continue
                        visited_edges.add(edge)
                        pts = graph[current][next_node]['pts']
                        o_current = graph.nodes[current]['o']
                        dist_start = np.linalg.norm(pts[0] - o_current)
                        dist_end = np.linalg.norm(pts[-1] - o_current)
                        if dist_start > dist_end:
                            pts = pts[::-1].copy()
                        poly = pts.tolist()
                        current = next_node
                        while current in graph and graph.degree[current] == 2:
                            neighbors = list(graph.neighbors(current))
                            next_cand = [n for n in neighbors if tuple(sorted((current, n))) not in visited_edges]
                            if len(next_cand) != 1:
                                break
                            next_node = next_cand[0]
                            edge = tuple(sorted((current, next_node)))
                            visited_edges.add(edge)
                            pts = graph[current][next_node]['pts']
                            o_current = graph.nodes[current]['o']
                            dist_start = np.linalg.norm(pts[0] - o_current)
                            dist_end = np.linalg.norm(pts[-1] - o_current)
                            if dist_start > dist_end:
                                pts = pts[::-1].copy()
                            # Append without duplicating endpoint
                            if np.allclose(poly[-1], pts[0]):
                                poly.extend(pts[1:].tolist())
                            else:
                                poly.extend(pts.tolist())
                            current = next_node
                        # Add the polyline if sufficient points
                        poly_array = np.array(poly)
                        if poly_array.shape[0] >= 3:
                            branch_polylines.append(poly_array)
           
            # Compute average bending energies
            branch_avg_bendings = []
            for c in branch_polylines:
                try:
                    avg_bend = average_bending_energy_curve(c, closed=False)
                    branch_avg_bendings.append(avg_bend)
                except ValueError as e:
                    print(f"Skipping branch in {file_name}: {e}")
           
            # Compute the overall average bending energy
            if len(branch_avg_bendings) > 0:
                avg_bending_energy = sum(branch_avg_bendings)
            else:
                avg_bending_energy = 0.0
            bending_energy_data[folder].append(avg_bending_energy)

# Compile features into a DataFrame and save to CSV


# Compile features into a DataFrame and save to CSV
data = []
for folder in folders:
    num_files = len(number_of_edges_data[folder])
    for i in range(num_files):
        row = {
            'id': id_data[folder][i],
            'label': label_map[folder],
            
            # Topological
            'N_edg': number_of_edges_data[folder][i],      
            'Betw': mean_betweenness_centrality_data[folder][i],          
            'Spec_ent': spectral_entropy_data[folder][i],     
            'Alg_Conn': algebraic_connectivity_data[folder][i],
            'Assort': assortativity_data[folder][i],           
            'G_diam': graph_diameter_data[folder][i],              
            
            # Geometric / Shape
            'Br_dens': branch_density_data[folder][i],          
            'Br_l': mean_branch_length_data[folder][i],       
            'Energy': bending_energy_data[folder][i],      
            'Apl': apl_data[folder][i],                      
            'Circ': circuity_data[folder][i],                
            'FD': fractal_dimension_data[folder][i],           
            
            # Directional
            'Dir_std': directional_std_data[folder][i],         
            'q1': mass_data[folder][1][i],                    
            'q2': mass_data[folder][2][i],                 
            'q3': mass_data[folder][3][i],                    
            'q4': mass_data[folder][4][i],                  
            'Dir_ent': directional_entropy_data[folder][i],           
            'Order': orientation_order_data[folder][i]                      
        }
        data.append(row)

# Create DataFrame

df = pd.DataFrame(data)
df.to_csv('main_astro_features.csv', index=False)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

...

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]